# KabulLearn Data Architecture

This notebook documents the current KabulLearn operational data model and proposes analytics/data-pipeline architecture for OLTP, OLAP, and review workflows.

Scope: KabulLearn only. Source schema reviewed from `prisma/schema.prisma`.

## 1. System Context

KabulLearn is a multilingual learning platform with students, educators, admins, courses, modules, lessons, quizzes, certificates, ratings, discussions, and review workflows.

Primary operational goals:

- Authenticate users and manage roles.
- Let educators create and submit courses.
- Let admins review, return, or publish courses.
- Let students enroll, consume lessons, complete quizzes, earn certificates, rate courses, and participate in discussion.
- Track video heartbeats and quiz attempts for progress and verification.

Primary analytics goals:

- Course engagement and completion.
- Quiz performance and learning outcomes.
- Instructor/course quality.
- Funnel metrics from registration to certificate.
- Content operations metrics for review and publishing.

## 2. Current OLTP Architecture

The current database is a PostgreSQL OLTP model managed through Prisma. It is normalized around application transactions: users, content hierarchy, progress events, submissions, and certificates.

High-level OLTP domains:

| Domain | Tables |
|---|---|
| Identity and auth | `User`, `Account`, `Session`, `VerificationToken`, `RateLimitBucket` |
| Creator profiles | `CreatorProfile`, `CourseInstructor` |
| Course content | `Course`, `Module`, `Lesson`, `Quiz`, `Question`, `AnswerChoice` |
| Learning activity | `Enrollment`, `UserProgress`, `LessonHeartbeat`, `QuizAttemptSession`, `QuizSubmission`, `QuizSubmissionAnswer` |
| Credentials | `Certificate` |
| Quality and review | `CourseRating`, `CourseReviewEvent`, `CourseReviewChecklistItem` |
| Community and notifications | `DiscussionThread`, `DiscussionReply`, `NotificationLog` |

## 3. OLTP Entities and Relationships

```text
User
  ├── Account / Session / VerificationToken
  ├── Enrollment ───────────── Course
  ├── UserProgress ─────────── Lesson ── Module ── Course
  ├── QuizSubmission ───────── Lesson/Quiz
  ├── Certificate ──────────── Course
  ├── CourseRating ─────────── Course
  ├── DiscussionThread/Reply ─ Course
  └── CreatorProfile / authoredCourses

Course
  ├── Module
  │   └── Lesson
  │       └── Quiz
  │           └── Question
  │               └── AnswerChoice
  ├── CourseReviewEvent
  ├── CourseReviewChecklistItem
  └── CourseInstructor ─────── CreatorProfile
```

## 4. OLTP Table Summary

| Table | Purpose | Important keys/indexes |
|---|---|---|
| `User` | People using the platform. Stores role, status, profile basics, password hash. | `email` unique; role/status fields |
| `CreatorProfile` | Public instructor profile separate from auth identity. | `username` unique; `userId` unique; `createdById` index |
| `CourseInstructor` | Many-to-many bridge between courses and creator profiles. | unique `(courseId, profileId)` |
| `Course` | Course container and review/publication state. | `slug` unique; indexes on `status`, `authorId`, `authorProfileId` |
| `Module` | Ordered course section. | unique `(courseId, order)`; index `courseId` |
| `Lesson` | Ordered learning unit. Type can be `VIDEO`, `READING`, or `QUIZ`. | unique `(moduleId, order)`; indexes `moduleId`, `type` |
| `Quiz` | One quiz shell per quiz lesson. | `lessonId` unique |
| `Question` | Ordered quiz question. | unique `(quizId, order)`; index `quizId` |
| `AnswerChoice` | Ordered choices for single/multiple-choice questions. | unique `(questionId, order)`; index `questionId` |
| `Enrollment` | Student-course membership. | unique `(userId, courseId)`; index `courseId` |
| `UserProgress` | Latest progress state for a user and lesson. | unique `(userId, lessonId)`; index `lessonId` |
| `LessonHeartbeat` | Video consumption telemetry. | indexes `(userId, lessonId, createdAt)`, `courseId` |
| `QuizAttemptSession` | Quiz attempt session/anti-replay window. | indexes `(userId, lessonId, startedAt)`, `courseId` |
| `QuizSubmission` | Quiz result event. | indexes `(userId, lessonId)`, `lessonId` |
| `QuizSubmissionAnswer` | Per-question selected/text answer. | unique `(submissionId, questionId, answerChoiceId)` |
| `Certificate` | Issued course certificate and verification code. | unique `(userId, courseId)`, `uuid`, `verificationCode` |
| `CourseRating` | One rating/comment per user per course. | unique `(userId, courseId)`; index `courseId` |
| `CourseReviewEvent` | Admin/editorial review history. | indexes `courseId`, `actorId` |
| `CourseReviewChecklistItem` | Review checklist state per course. | index `courseId` |
| `DiscussionThread` | Course-level discussion topic. | indexes `courseId`, `authorId` |
| `DiscussionReply` | Reply to thread. | indexes `threadId`, `authorId` |
| `NotificationLog` | Email/notification dispatch log. | indexes `email`, `status` |
| `RateLimitBucket` | Rate-limit counters. | primary key `key` |

## 5. OLTP Transaction Flows

### Student flow

1. `User` is created with `UserRole.STUDENT` and `UserStatus.VERIFICATION_PENDING` or `ACTIVE`.
2. Email verification uses `VerificationToken` and `NotificationLog`.
3. Student enrolls: insert into `Enrollment`.
4. Student opens lessons: `UserProgress` is created/updated.
5. Video lessons: `LessonHeartbeat` rows capture playback progress.
6. Quiz lessons: `QuizAttemptSession` starts, `QuizSubmission` and `QuizSubmissionAnswer` record outcome.
7. Passing module quizzes updates `UserProgress`.
8. Completing course creates or updates `Certificate`.
9. Optional feedback creates `CourseRating` and discussions.

### Educator/admin flow

1. Educator creates `Course`, `Module`, `Lesson`, `Quiz`, `Question`, `AnswerChoice`.
2. Course is submitted: `Course.status = PENDING_REVIEW`, `CourseReviewEvent(type=SUBMITTED)`.
3. Admin reviews checklist: `CourseReviewChecklistItem`.
4. Admin publishes or returns: `CourseReviewEvent(type=PUBLISHED|RETURNED)`, `Course.status` changes.
5. Published courses become available to students.

## 6. Proposed OLAP Architecture

Keep OLTP as the source of truth. Build analytics tables from OLTP snapshots/events into a warehouse-style schema.

Recommended approach for current scale:

- Start with a Postgres analytics schema, for example `analytics`.
- Use scheduled SQL/dbt-style transforms from `public` OLTP tables.
- Move to a separate warehouse only if volume or BI concurrency grows.

Core grain choices:

| Fact table | Grain |
|---|---|
| `fact_enrollment` | one row per user-course enrollment |
| `fact_lesson_progress` | one row per user-lesson progress record |
| `fact_video_heartbeat` | one row per heartbeat event, or aggregated per user-lesson-day |
| `fact_quiz_submission` | one row per quiz submission |
| `fact_quiz_answer` | one row per submitted question answer |
| `fact_certificate` | one row per issued certificate |
| `fact_course_rating` | one row per rating |
| `fact_course_review_event` | one row per review event |
| `fact_discussion_activity` | one row per thread/reply event |

## 7. Proposed Dimensions

| Dimension | Source tables | Notes |
|---|---|---|
| `dim_user` | `User` | Keep PII minimal. Hash or omit email in analytics where possible. |
| `dim_course` | `Course`, `CreatorProfile` | Include status, level, author profile, published/submitted dates. |
| `dim_module` | `Module`, `Course` | Course-module hierarchy. |
| `dim_lesson` | `Lesson`, `Module`, `Course` | Include `lesson_type`, `is_final_test`, order fields. |
| `dim_quiz` | `Quiz`, `Lesson` | One quiz per quiz lesson. |
| `dim_question` | `Question`, `Quiz` | Include question type and order. |
| `dim_date` | generated calendar | Useful for daily/weekly/monthly BI. |
| `dim_creator` | `CreatorProfile` | Public instructor analytics. |

## 8. OLAP Star Schema Draft

```text
dim_user ─────────────┐
dim_course ───────┐   ├── fact_enrollment
dim_date ─────────┘   └── fact_certificate

dim_user ─────────────┐
dim_course ───────────┤
dim_module ───────────┤── fact_lesson_progress
dim_lesson ───────────┘

dim_user ─────────────┐
dim_course ───────────┤
dim_module ───────────┤── fact_quiz_submission ── fact_quiz_answer
dim_lesson ───────────┤            │                    │
dim_quiz ─────────────┘            └── dim_question ────┘

dim_course ───────────┐
dim_creator ──────────┤── fact_course_review_event
dim_user(actor) ──────┘
```

## 9. Example Analytics Tables

```sql
CREATE SCHEMA IF NOT EXISTS analytics;

CREATE TABLE IF NOT EXISTS analytics.dim_course (
  course_key text PRIMARY KEY,
  slug text,
  status text,
  level text,
  title_en text,
  title_ps text,
  title_da text,
  author_user_id text,
  author_profile_id text,
  submitted_at timestamptz,
  published_at timestamptz,
  created_at timestamptz,
  updated_at timestamptz
);

CREATE TABLE IF NOT EXISTS analytics.dim_lesson (
  lesson_key text PRIMARY KEY,
  module_key text NOT NULL,
  course_key text NOT NULL,
  lesson_order int,
  lesson_type text,
  title_en text,
  title_ps text,
  title_da text,
  is_final_test boolean,
  passing_score int,
  created_at timestamptz,
  updated_at timestamptz
);

CREATE TABLE IF NOT EXISTS analytics.fact_quiz_submission (
  submission_key text PRIMARY KEY,
  user_key text NOT NULL,
  course_key text NOT NULL,
  module_key text NOT NULL,
  lesson_key text NOT NULL,
  score numeric,
  passed boolean,
  duration_ms int,
  started_at timestamptz,
  submitted_at timestamptz
);
```

## 10. Pipeline Architecture Options

### Option A: Simple scheduled transforms inside Postgres

Best for current small/medium scale.

```text
Application OLTP tables
        │
        ├── scheduled SQL transforms
        ▼
analytics.dim_* and analytics.fact_*
        │
        ▼
dashboard / notebook / BI reports
```

Pros: simple, cheap, easy to audit.  
Cons: analytical queries share Postgres resources unless moved to replica/warehouse.

### Option B: Incremental ETL/ELT to warehouse

```text
OLTP Postgres
   │
   ├── extracts by updated_at / created_at
   ▼
staging tables
   │
   ├── dbt/SQL transforms
   ▼
warehouse marts
```

Pros: stronger reporting performance and governance.  
Cons: more infrastructure and orchestration.

### Option C: Event stream for learning telemetry

```text
Client/API events
   │
   ├── validation/rate limit
   ▼
event queue or append-only event table
   │
   ├── consumers
   ▼
OLTP progress + analytics facts
```

Pros: best for high-volume heartbeats and clickstream.  
Cons: more moving parts; not needed until event volume grows.

## 11. Recommended First Version

Use a hybrid OLTP + analytics schema in the same Postgres project, preferably on a read replica if available.

Recommended steps:

1. Keep Prisma OLTP schema unchanged for application correctness.
2. Add an `analytics` schema for materialized dimensions and facts.
3. Build daily/hourly transforms for low-volume entities: users, courses, modules, lessons, enrollments, submissions, certificates, ratings.
4. Aggregate `LessonHeartbeat` by user-course-lesson-day to avoid huge raw BI scans.
5. Keep raw heartbeat rows in OLTP for operational verification, but report from aggregated facts.
6. Add BI dashboards only after the fact tables are stable.

Suggested refresh cadence:

| Data | Cadence |
|---|---|
| Course/content dimensions | hourly or on publish |
| Enrollment/progress/submissions | hourly |
| Heartbeat aggregates | hourly/daily |
| Certificates | hourly |
| Ratings/discussions | daily or hourly |

## 12. Key Metrics

| Metric | Source | Notes |
|---|---|---|
| Registered users | `User` | Segment by role/status. |
| Active learners | `UserProgress`, `LessonHeartbeat`, `QuizSubmission` | Activity in last 7/30 days. |
| Enrollments | `Enrollment` | User-course grain. |
| Course starts | first `UserProgress` per course | Join lesson -> module -> course. |
| Course completions | `Certificate` or all module quiz progress | Certificate is the cleanest business event. |
| Completion rate | certificates / enrollments | Filter published courses. |
| Average quiz score | `QuizSubmission.score` | Segment by course/module/lesson. |
| Quiz pass rate | `QuizSubmission.passed` | Submission grain; also use latest attempt. |
| Video consumption | `LessonHeartbeat.consumedPct` | Aggregate max consumed pct per user-lesson. |
| Rating average | `CourseRating.rating` | Course and instructor level. |
| Review cycle time | `CourseReviewEvent` | submitted -> published/returned. |

## 13. Data Quality Checks

Useful checks to run before analytics refresh:

- Every `Lesson(type='QUIZ')` should have one `Quiz`.
- Every `Quiz` should point to a `Lesson(type='QUIZ')`.
- Every `Module` should have stable unique lesson order.
- Every `Course(status='PUBLISHED')` should have at least one module and lesson.
- Published courses should have at least one final/required quiz per module if that is a business rule.
- `Lesson(type='READING')` should not require `youtubeUrl`.
- `Lesson(type='VIDEO')` should have a valid `youtubeUrl`.
- Certificate rows should only exist for enrolled users who passed required modules.
- Avoid duplicate analytics records by preserving OLTP primary keys as natural keys in facts/dimensions.

Example check:

```sql
SELECT l.id, l.titleEn
FROM "Lesson" l
LEFT JOIN "Quiz" q ON q."lessonId" = l.id
WHERE l.type = 'QUIZ' AND q.id IS NULL;
```

## 14. Privacy and Governance

Analytics should minimize personally identifiable information.

Recommendations:

- Use `userId` as analytics key; avoid copying raw emails to facts.
- If email is needed for admin reports, keep it in restricted dimensions only.
- Treat quiz answers and discussion text as sensitive learning data.
- Keep operational auth tables (`Session`, `Account`, password hashes) out of analytics.
- Keep `RateLimitBucket` operational only unless security analytics are explicitly needed.
- For public dashboards, aggregate to course/module level and suppress small counts.

## 15. OLTP ERD Diagram

This ERD summarizes the current normalized operational schema. It focuses on application transactions and preserves the content hierarchy, identity, progress, quiz, review, and community relationships.

```mermaid
erDiagram
  User ||--o{ Account : has
  User ||--o{ Session : has
  User ||--o{ Enrollment : enrolls
  User ||--o{ UserProgress : tracks
  User ||--o{ QuizSubmission : submits
  User ||--o{ Certificate : earns
  User ||--o{ CourseRating : rates
  User ||--o{ LessonHeartbeat : watches
  User ||--o{ QuizAttemptSession : starts
  User ||--o{ DiscussionThread : authors
  User ||--o{ DiscussionReply : writes
  User ||--o{ CourseReviewEvent : acts
  User ||--o{ Course : authors
  User ||--o{ CreatorProfile : creates
  User ||--o| CreatorProfile : owns_profile

  CreatorProfile ||--o{ CourseInstructor : listed_on
  CreatorProfile ||--o{ Course : public_author
  Course ||--o{ CourseInstructor : has

  Course ||--o{ Module : contains
  Module ||--o{ Lesson : contains
  Lesson ||--o| Quiz : may_have
  Quiz ||--o{ Question : contains
  Question ||--o{ AnswerChoice : has

  Course ||--o{ Enrollment : receives
  Lesson ||--o{ UserProgress : measured_by
  Lesson ||--o{ LessonHeartbeat : receives
  Lesson ||--o{ QuizAttemptSession : attempted_as
  Lesson ||--o{ QuizSubmission : submitted_for
  QuizSubmission ||--o{ QuizSubmissionAnswer : includes
  Question ||--o{ QuizSubmissionAnswer : answered
  AnswerChoice ||--o{ QuizSubmissionAnswer : selected

  Course ||--o{ Certificate : issues
  Course ||--o{ CourseRating : receives
  Course ||--o{ CourseReviewEvent : reviewed_by
  Course ||--o{ CourseReviewChecklistItem : checked_by
  Course ||--o{ DiscussionThread : discusses
  DiscussionThread ||--o{ DiscussionReply : has
```

Notes:

- `Course.authorId` uses `onDelete: Restrict`, so authors with courses cannot be deleted unless courses are reassigned or removed.
- Most child content under `Course`, `Module`, `Lesson`, and quiz tables cascades on delete.
- `CreatorProfile` allows public instructor identity to be separated from login identity.
- `Lesson.type` controls whether a lesson behaves as video, reading, or quiz content.

## 16. OLAP ERD Diagram

This ERD is a proposed analytics/star-schema design. It denormalizes the current OLTP model into dimensions and facts that are easier to query for dashboards and reporting.

```mermaid
erDiagram
  dim_user ||--o{ fact_enrollment : user_key
  dim_course ||--o{ fact_enrollment : course_key
  dim_date ||--o{ fact_enrollment : enrolled_date_key

  dim_user ||--o{ fact_lesson_progress : user_key
  dim_course ||--o{ fact_lesson_progress : course_key
  dim_module ||--o{ fact_lesson_progress : module_key
  dim_lesson ||--o{ fact_lesson_progress : lesson_key
  dim_date ||--o{ fact_lesson_progress : updated_date_key

  dim_user ||--o{ fact_video_engagement : user_key
  dim_course ||--o{ fact_video_engagement : course_key
  dim_module ||--o{ fact_video_engagement : module_key
  dim_lesson ||--o{ fact_video_engagement : lesson_key
  dim_date ||--o{ fact_video_engagement : activity_date_key

  dim_user ||--o{ fact_quiz_submission : user_key
  dim_course ||--o{ fact_quiz_submission : course_key
  dim_module ||--o{ fact_quiz_submission : module_key
  dim_lesson ||--o{ fact_quiz_submission : lesson_key
  dim_quiz ||--o{ fact_quiz_submission : quiz_key
  dim_date ||--o{ fact_quiz_submission : submitted_date_key

  fact_quiz_submission ||--o{ fact_quiz_answer : submission_key
  dim_question ||--o{ fact_quiz_answer : question_key

  dim_user ||--o{ fact_certificate : user_key
  dim_course ||--o{ fact_certificate : course_key
  dim_date ||--o{ fact_certificate : issued_date_key

  dim_user ||--o{ fact_course_rating : user_key
  dim_course ||--o{ fact_course_rating : course_key
  dim_date ||--o{ fact_course_rating : rating_date_key

  dim_course ||--o{ fact_course_review_event : course_key
  dim_user ||--o{ fact_course_review_event : actor_user_key
  dim_date ||--o{ fact_course_review_event : event_date_key

  dim_course ||--o{ dim_module : contains
  dim_module ||--o{ dim_lesson : contains
  dim_lesson ||--o| dim_quiz : has
  dim_quiz ||--o{ dim_question : contains
  dim_creator ||--o{ dim_course : teaches
```

Suggested fact grains:

| Fact | Grain |
|---|---|
| `fact_enrollment` | one row per user-course enrollment |
| `fact_lesson_progress` | one row per user-lesson current progress state |
| `fact_video_engagement` | one row per user-lesson-day aggregate, not every heartbeat |
| `fact_quiz_submission` | one row per quiz attempt/submission |
| `fact_quiz_answer` | one row per submitted question answer |
| `fact_certificate` | one row per issued certificate |
| `fact_course_rating` | one row per user-course rating |
| `fact_course_review_event` | one row per review workflow event |

## 17. Open Decisions

- Should analytics live in the same Neon/Postgres project or a separate warehouse/read replica?
- What is the official definition of course completion: certificate issued, all quizzes passed, or both?
- Should video `LessonHeartbeat` remain raw forever, or be archived after aggregation?
- Do we need language-specific analytics for English/Pashto/Dari course content consumption?
- Should course review metrics be visible to educators, admins only, or both?
- What BI tool will consume the analytics tables?